In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score


url = "https://raw.githubusercontent.com/mohitgupta-omg/Kaggle-SMS-Spam-Collection-Dataset-/master/spam.csv"
df = pd.read_csv(url, encoding='latin-1')

# Clean columns
df = df[['v1', 'v2']]
df.columns = ['label', 'message']
df['label'] = df['label'].map({'ham': 0, 'spam': 1})

# Vectorize: Keep top 1500 words
vectorizer = CountVectorizer(stop_words='english', max_features=1500)
X_raw = vectorizer.fit_transform(df['message']).toarray()
y_raw = df['label'].values


X_train, X_test, y_train, y_test = train_test_split(X_raw, y_raw, test_size=0.2, random_state=42)


X_train_tensor = torch.FloatTensor(X_train)
y_train_tensor = torch.FloatTensor(y_train).unsqueeze(1)
X_test_tensor = torch.FloatTensor(X_test)
y_test_tensor = torch.FloatTensor(y_test).unsqueeze(1)

print(f"Dataset Loaded. Total Messages: {len(df)}")


class SpamDetector(nn.Module):
    def __init__(self, input_size):
        super(SpamDetector, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 512),
            nn.ReLU(),
            nn.Dropout(0.5),      # Drop 50% of neurons to prevent memorizing
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 1)     # Output: 1 number (Spam score)
        )

    def forward(self, x):
        return self.net(x)


model = SpamDetector(input_size=1500)


criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("\nStarting CPU Training...")
for epoch in range(50):
    model.train()

    # 1. Forward
    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)

    # 2. Backward
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch+1) % 10 == 0:
        print(f'Epoch [{epoch+1}/50], Loss: {loss.item():.4f}')


model.eval()
with torch.no_grad():
    y_pred_logits = model(X_test_tensor)
    y_pred = (torch.sigmoid(y_pred_logits) > 0.5).float()
    acc = accuracy_score(y_test, y_pred)
    print(f"\nFinal Test Accuracy: {acc*100:.2f}%")

# Custom Test Function
def is_spam(text):
    vec = vectorizer.transform([text]).toarray()
    vec_tensor = torch.FloatTensor(vec)
    pred = torch.sigmoid(model(vec_tensor)).item()
    return "SPAM 🚨" if pred > 0.5 else "Safe ✅"

print("\n--- Interactive Test ---")
print(f"Msg: 'URGENT! You won $500 cash' -> {is_spam('URGENT! You won $500 cash')}")
print(f"Msg: 'Hey man, are we still going to the gym?' -> {is_spam('Hey man, are we still going to the gym?')}")

Dataset Loaded. Total Messages: 5572

Starting CPU Training...
Epoch [10/50], Loss: 0.4743
Epoch [20/50], Loss: 0.1882
Epoch [30/50], Loss: 0.0808
Epoch [40/50], Loss: 0.0296
Epoch [50/50], Loss: 0.0149

Final Test Accuracy: 98.39%

--- Interactive Test ---
Msg: 'URGENT! You won $500 cash' -> SPAM 🚨
Msg: 'Hey man, are we still going to the gym?' -> Safe ✅
